# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² dataset using the `mlcroissant` library and Croissant schema.

### Dataset Source
The dataset source is defined via the following Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Install mlcroissant if it's not already installed
!pip install mlcroissant

## 1. Data Loading
Load the FAIR² metadata and available records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset using Croissant
dataset = mlc.Dataset(croissant_url)

# Print metadata summary
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` identifiers.

We'll enumerate the available record sets in the dataset, and for each, print its `@id`, `name`, and referenced fields and their IDs.

In [ ]:
# List all record sets in the dataset
record_sets = list(dataset.metadata.record_sets)

print(f"Found {len(record_sets)} record set(s) in the dataset.")

for rs in record_sets:
    print(f"\nRecord set: {rs['@id']}")
    print(f"  Name: {rs.get('name', 'N/A')}")
    # List all fields in the record set
    print("  Fields and IDs:")
    if 'fields' in rs:
        for field in rs['fields']:
            print(f"    - {field['@id']} (name: {field.get('name', 'N/A')})")
    else:
        print("    No fields found.")

## 3. Data Extraction
Select one or more record sets and load them into Pandas DataFrames for analysis. Reference each record set and field by their `@id`.

The record set `@id`s and their field `@id`s discovered in the previous step are used here.

In [ ]:
# List all record set @id values
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for rs_id in record_set_ids:
    print(f"\nLoading data for record set: {rs_id}")
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded {len(df)} records.")
            print(f"Columns: {df.columns.tolist()}")
        else:
            print("No records found.")
    except Exception as e:
        print(f"  Could not load records (error: {e}).")

# Display the first DataFrame (if any)
if dataframes:
    first_rs_id = next(iter(dataframes))
    print(f"\nSample records from record set {first_rs_id}:")
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply typical preprocessing: filtering out values, normalizing a numeric field, grouping by a key attribute.

Adjust the `numeric_field` and `group_field` below to use the appropriate `@id`s from your dataset.

In [ ]:
# -- Example: EDA for the first available DataFrame --
import numpy as np

if dataframes:
    # Choose record set and fields (edit as appropriate)
    record_set_id = first_rs_id
    df = dataframes[record_set_id]
    print(f"EDA on record set: {record_set_id}")
    
    # Inspect a few columns for numeric fields
    numeric_field = None
    for col in df.columns:
        if np.issubdtype(df[col].dtype, np.number):
            numeric_field = col
            print(f"Using numeric field for EDA: {numeric_field}")
            break
    if numeric_field is None:
        print("No numeric field found for EDA.")
    else:
        # Filtering example
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records where {numeric_field} > mean:")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
            filtered_df[numeric_field].std()
        )
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by first non-numeric field
        group_field = None
        for col in df.columns:
            if not np.issubdtype(df[col].dtype, np.number):
                group_field = col
                print(f"Grouping by field: {group_field}")
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame(name=f"mean_{numeric_field}")
            print(f"Grouped means of {numeric_field} by {group_field}:")
            display(grouped_df.head())
        else:
            print("No non-numeric field for grouping found.")
else:
    print("No DataFrames available for EDA.")

## 5. Visualization
Visualize distributions of numeric fields or relationships between variables.

The following example uses matplotlib and seaborn to plot a numeric field's distribution and a boxplot by group (if grouping field exists).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if dataframes and numeric_field is not None:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field], bins=20, kde=True, color='skyblue')
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Optional: boxplot by group_field
    if group_field:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR² dataset using the Croissant schema and the `mlcroissant` library. We inspected the metadata, listed record sets and their available fields, loaded data for exploration and performed basic EDA including filtering, normalization, grouping, and visualization.

Further analysis or modeling can be performed according to your research questions using the structured DataFrames extracted above.